# 02 — Évaluation du modèle pré-entraîné (baseline)

**Rôle** : faire tourner TorchXRayVision sur les splits, mettre en cache les scores bruts, mesurer l'AUROC avec intervalles de confiance.

**Entrées** : `splits.json` (notebook 01). **Sorties** : `raw_scores.npz` (scores bruts par image), courbes ROC.

> ⚠️ **Fuite de données.** Les poids `-chex` ont été entraînés sur CheXpert, les poids `-nih` sur NIH, les poids `-all` sur presque tout. Évaluer un modèle sur le dataset qui a servi à l'entraîner donne des résultats trop beaux. Avec `-chex`, évaluez sur **NIH** (jamais vu). C'est aussi une démonstration de décalage de distribution.

In [ ]:
# --- Initialisation (identique dans tous les notebooks) ---
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    # Internet doit être ACTIVÉ dans les réglages du notebook Kaggle.
    # Remplacez VOTRE_COMPTE par votre identifiant GitHub (dépôt medicure-ai).
    %pip install -q git+https://github.com/VOTRE_COMPTE/medicure-ai.git torchxrayvision
else:
    sys.path.insert(0, str(Path.cwd().parent))  # notebook lancé depuis le dossier notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from medguard.config import TARGETS, MODEL_WEIGHTS, Settings, load_settings, save_settings
from medguard.paths import find_file

# --- Configuration (à adapter) ---
DATASET = os.environ.get("MEDGUARD_DATASET", "nih")            # "nih" ou "chexpert"
DATA_ROOT = Path(os.environ.get("MEDGUARD_DATA", "/kaggle/input/data"))  # À ADAPTER au dataset attaché
WEIGHTS = os.environ.get("MEDGUARD_WEIGHTS", MODEL_WEIGHTS)
N_MAX = int(os.environ.get("MEDGUARD_NMAX", 6000))               # nb max d'images chargées
SEED = 42

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("MEDGUARD_WORK", "../outputs")).resolve()
ARTIFACTS = WORK / "artifacts"
FIGURES = WORK / "figures"
for d in (ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle :", IN_KAGGLE, "| dataset :", DATASET, "| poids :", WEIGHTS, "| sortie :", WORK)

In [ ]:
from medguard.data import load_dataset, predict_paths
from medguard.model import get_predictor
from medguard.splits import load_splits
from medguard.evaluation import auroc, bootstrap_ci

In [ ]:
df, EVAL_TARGETS = load_dataset(DATASET, DATA_ROOT, n_max=N_MAX, seed=SEED)
splits = load_splits(find_file("splits.json", [WORK, ARTIFACTS]))
id_to_split = {i: name for name, ids in splits.items() for i in ids}
df["split"] = df.image_id.astype(str).map(id_to_split)
df = df.dropna(subset=["split"]).reset_index(drop=True)
print(df.split.value_counts().to_dict())

predictor = get_predictor(WEIGHTS)
print(predictor.info)

# Garde-fou contre la fuite de données
trained_on = {"chex": "chexpert", "nih": "nih"}
for key, ds in trained_on.items():
    if WEIGHTS.endswith(key) and DATASET == ds:
        print(f"\n*** ATTENTION : les poids '{WEIGHTS}' ont été entraînés sur {ds}. Évaluation NON externe. ***")
if WEIGHTS.endswith("all"):
    print("\n*** ATTENTION : les poids '-all' ont vu la plupart des datasets publics : évaluation probablement optimiste. ***")

## Inférence (mise en cache)

In [ ]:
scores = predict_paths(predictor, df.path.tolist(), batch_size=32)
np.savez(ARTIFACTS / "raw_scores.npz",
         ids=np.array(df.image_id.astype(str).tolist()), split=np.array(df.split.astype(str).tolist()),
         scores=scores, targets=np.array(list(predictor.targets)))
print("Écrit :", ARTIFACTS / "raw_scores.npz", scores.shape)

## AUROC par pathologie (split test, IC à 95 % par bootstrap)

In [ ]:
names = list(predictor.targets)
test = (df.split == "test").values
rows = []
fig, axes = plt.subplots(1, len(EVAL_TARGETS), figsize=(4.2 * len(EVAL_TARGETS), 3.8))
axes = np.atleast_1d(axes)
for ax, t in zip(axes, EVAL_TARGETS):
    y = df[t].values.astype(float); s = scores[:, names.index(t)]
    ok = test & ~np.isnan(y)
    est, lo, hi = bootstrap_ci(auroc, y[ok], s[ok], n_boot=500)
    rows.append({"pathologie": TARGETS.get(t, t), "n": int(ok.sum()), "positifs": int(y[ok].sum()), "AUROC": est, "IC95 bas": lo, "IC95 haut": hi})
    # courbe ROC
    order = np.argsort(-s[ok]); yy = y[ok][order]
    tpr = np.cumsum(yy) / max(yy.sum(), 1); fpr = np.cumsum(1 - yy) / max((1 - yy).sum(), 1)
    ax.plot(fpr, tpr, color="#4b7bec"); ax.plot([0, 1], [0, 1], "--", color="grey")
    ax.set_title(f"{TARGETS.get(t, t)}\nAUROC {est:.3f} [{lo:.3f}–{hi:.3f}]"); ax.set_xlabel("1 − spécificité"); ax.set_ylabel("sensibilité")
plt.tight_layout(); plt.savefig(FIGURES / "02_roc.png", dpi=150); plt.show()
pd.DataFrame(rows).round(3)

### Lecture
- Un AUROC élevé prouve que le modèle **classe** bien les cas, pas que ses scores sont des **probabilités** : ils sont normalisés autour d'un point de fonctionnement. C'est l'objet du notebook 03.
- Un AUROC nettement plus bas sur un dataset externe que celui annoncé dans la littérature est normal (décalage de distribution) : à présenter comme une limite honnête.